# Member 3 — Classification A2 (10-algorithm comparison) and A3 (final model selection)


Every model was trained on the same Adult split (36,140 train / 9,035 test rows, `adult_prep.py`) with the minority
class up-weighted, so the numbers are directly comparable. Sources (run these first):

| Part | Algorithms | Owner | Results file |
|---|---|---|---|
| A (Review 1) | Logistic Regression, KNN, Gaussian NB, Decision Tree, SVC | Review 1 | `model_comparison.csv` |
| B | Random Forest, AdaBoost | Member 1 | `partB_member1_model_comparison.csv` |
| B | Bagging, MLP | Member 2 | `partB_member2_model_comparison.csv` |
| B | Gradient Boosting | Member 3 | `partB_member3_model_comparison.csv` |

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from IPython.display import display
from sklearn.metrics import roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report

from adult_prep import prepare

sns.set_theme(style='whitegrid', palette='colorblind')
warnings.filterwarnings('ignore', category=UserWarning)
os.environ['PYTHONWARNINGS'] = 'ignore::UserWarning'

RESULTS_DIR = Path('../results/classification')
MODELS_DIR = Path('../models/classification')
METRICS = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'roc_auc', 'f1_macro', 'balanced_accuracy']

def save_fig(name):
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / name, dpi=150, bbox_inches='tight')

: 

## A2. Consolidated comparison of all ten classifiers

The four result files share one column layout, so they are stacked directly. The majority-class baseline is kept
as a reference line, not as an algorithm.

In [ ]:
SOURCES = [('model_comparison.csv', 'A', 'Review 1'),
           ('partB_member1_model_comparison.csv', 'B', 'Member 1'),
           ('partB_member2_model_comparison.csv', 'B', 'Member 2'),
           ('partB_member3_model_comparison.csv', 'B', 'Member 3')]
frames = []
for file, part, owner in SOURCES:
    path = RESULTS_DIR / file
    if not path.exists():
        raise FileNotFoundError(f'{file} is missing - run the notebook that produces it first.')
    frames.append(pd.read_csv(path).assign(part=part, owner=owner))
allm = pd.concat(frames, ignore_index=True).set_index('model')

baseline = allm.loc['Majority-class baseline', METRICS].astype(float)
table = allm.drop(index='Majority-class baseline')[['part', 'owner'] + METRICS]
assert table.index.is_unique and len(table) == 10, f'expected 10 distinct classifiers, found {len(table)}'
table = table.sort_values('f1_weighted', ascending=False)
table.insert(0, 'rank (weighted F1)', range(1, 11))
table.to_csv(RESULTS_DIR / 'all_classifiers_comparison.csv')

print('majority-class baseline:', baseline.round(3).to_dict())
display(table.round(4).style.format(precision=4).highlight_max(subset=METRICS, color='#cfe8cf'))

**Observation (A2).**
- **Ensembles lead.** Bagging is first on weighted F1 (0.849) and accuracy (0.846). Gradient Boosting is second on
  weighted F1 (0.845) but first on ROC-AUC (0.926), macro F1 (0.804), balanced accuracy (0.844) and weighted precision.
  Four of the top five models are Part B ensembles; KNN (0.836) is the best Part A model.
- **Every model beats the majority-class baseline** (weighted F1 0.646, ROC-AUC 0.5) **except Gaussian Naive Bayes**
  on weighted F1 (0.605). Its ROC-AUC (0.834) shows it still ranks people better than chance, but its independence
  assumption fails on correlated one-hot features, so its hard predictions are poor.
- **The MLP** is the weakest Part B model on weighted F1 (0.814) but still has a high ROC-AUC (0.908): it ranks well
  and leans towards predicting `>50K`.
- Weighted recall equals accuracy for every model — expected for a binary problem.

In [ ]:
plot_df = table[['accuracy', 'f1_weighted', 'f1_macro', 'roc_auc']].iloc[::-1]
ax = plot_df.plot.barh(figsize=(10, 8), width=0.85)
ax.axvline(baseline['f1_weighted'], color='grey', ls=':', label=f"baseline weighted F1 ({baseline['f1_weighted']:.3f})")
ax.set_xlim(0.55, 0.95)
ax.set_title('All ten classifiers on the same Adult test set')
ax.set_xlabel('score')
ax.set_ylabel('classifier')
ax.legend(title='metric', loc='lower right')
save_fig('all_classifiers_comparison.png')
plt.show()

## A3. Final model selection

### A3.1 Shortlist

The two strongest models are compared on every metric, including the two that show minority-class performance
(macro F1 and balanced accuracy) and their 5-fold CV results.

In [ ]:
def load_cv():
    """Stack the CV result files, normalising the Review 1 column names to the Part B ones."""
    part_a = pd.read_csv(RESULTS_DIR / 'cv_results.csv').rename(columns={
        'CV accuracy': 'cv_accuracy_mean', 'CV accuracy std': 'cv_accuracy_std',
        'CV weighted F1': 'cv_f1_weighted_mean', 'CV weighted F1 std': 'cv_f1_weighted_std',
        'CV ROC-AUC': 'cv_roc_auc_mean', 'CV ROC-AUC std': 'cv_roc_auc_std'}).drop(columns=['test weighted F1'])
    parts = [part_a] + [pd.read_csv(RESULTS_DIR / f'partB_member{i}_cv_results.csv') for i in (1, 2, 3)]
    return pd.concat(parts, ignore_index=True).set_index('model')

cv_all = load_cv()
display(cv_all.round(4))

In [ ]:
SHORTLIST = ['Bagging', 'Gradient Boosting']
short = table.loc[SHORTLIST, METRICS].T
short['difference (GB - Bagging)'] = short['Gradient Boosting'] - short['Bagging']
display(short.round(4))
display(cv_all.loc[SHORTLIST].round(4))

**Observation.** The two candidates split the metrics:
- **Bagging** is ahead on accuracy (+0.009) and weighted F1 (+0.005). Both are dominated by the 75% `<=50K` class.
- **Gradient Boosting** is ahead on ROC-AUC (+0.013 on test, and 0.927 vs 0.914 in 5-fold CV — about four fold-to-fold
  standard deviations apart), balanced accuracy (+0.026) and weighted precision (+0.012); macro F1 is tied.
- The CV results confirm both rankings are stable: Bagging's weighted-F1 lead (0.850 vs 0.842 in CV) and Gradient
  Boosting's ROC-AUC lead both hold across folds.

### A3.2 Class-level behaviour of the two candidates

The saved tuned models are reloaded and scored on the same test split to see *where* they differ.

In [ ]:
data = prepare('../data/raw/adult')
X_test, y_test = data['X_test'], data['y_test']
class_names = list(data['target_le'].classes_)
cands = {name: joblib.load(MODELS_DIR / f"{name.lower().replace(' ', '_')}.pkl") for name in SHORTLIST}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes[:2], cands.items()):
    pred = model.predict(X_test)
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=class_names).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.grid(False)
    ax.set_title(f'{name}: confusion matrix')
    ax.set_xlabel('predicted income class')
    ax.set_ylabel('true income class')
    print(f'--- {name} ---')
    print(classification_report(y_test, pred, target_names=class_names, digits=3))
for name, model in cands.items():
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    axes[2].plot(fpr, tpr, label=f"{name} (AUC = {table.loc[name, 'roc_auc']:.3f})")
axes[2].plot([0, 1], [0, 1], 'k--', label='chance')
axes[2].set_title('ROC curves of the two candidates')
axes[2].set_xlabel('false positive rate')
axes[2].set_ylabel('true positive rate (recall of >50K)')
axes[2].legend(loc='lower right')
save_fig('final_model_candidates.png')
plt.show()

**Observation.** The difference is in the minority class (2,240 `>50K` people in the test set):
- Gradient Boosting catches **86%** of high earners (recall 0.858) against Bagging's 76% — roughly 215 more `>50K`
  people found.
- The cost is about 300 more false alarms: Gradient Boosting's `>50K` precision is 0.625 against Bagging's 0.665,
  and its `<=50K` recall drops from 0.874 to 0.830.
- The ROC curves show Gradient Boosting above Bagging over almost the whole range, i.e. it ranks people better at
  any threshold, not just at the default 0.5.

### A3.3 Tuning evidence for all ten models

Each model's tuned settings and the documented change from its baseline (test set). Gaussian NB and Logistic
Regression have no tuned hyperparameters in Review 1 and are listed without a change.

In [ ]:
a_tune = pd.read_csv(RESULTS_DIR / 'tuning_results.csv').rename(columns={
    'test_f1_before': 'f1_weighted base', 'test_f1_after': 'f1_weighted tuned',
    'test_roc_auc_before': 'roc_auc base', 'test_roc_auc_after': 'roc_auc tuned'})[
    ['model', 'best_params', 'f1_weighted base', 'f1_weighted tuned', 'roc_auc base', 'roc_auc tuned']]
b_rows = []
for i in (1, 2, 3):
    long = pd.read_csv(RESULTS_DIR / f'partB_member{i}_tuning_results.csv')
    for model, grp in long.groupby('model', sort=False):
        g = grp.set_index('metric')
        b_rows.append({'model': model, 'best_params': g['best_params'].iloc[0],
                       'f1_weighted base': g.loc['f1_weighted', 'base'], 'f1_weighted tuned': g.loc['f1_weighted', 'tuned'],
                       'roc_auc base': g.loc['roc_auc', 'base'], 'roc_auc tuned': g.loc['roc_auc', 'tuned']})
tuning = pd.concat([a_tune, pd.DataFrame(b_rows)], ignore_index=True).set_index('model')
tuning['f1_weighted change'] = tuning['f1_weighted tuned'] - tuning['f1_weighted base']
tuning['roc_auc change'] = tuning['roc_auc tuned'] - tuning['roc_auc base']
untuned = [m for m in table.index if m not in tuning.index]
print('models without a tuning step in their notebook:', untuned)
display(tuning.round(4))
tuning.to_csv(RESULTS_DIR / 'all_classifiers_tuning_summary.csv')

**Observation (tuning applied and improvement documented).**
- **8 of the 10 models** were tuned with `GridSearchCV` (Logistic Regression and Gaussian NB were used with fixed
  settings in Review 1).
- Largest weighted-F1 gains: Gradient Boosting **+0.016**, Bagging +0.011, KNN +0.007. Bagging also gained +0.022
  ROC-AUC.
- Tuning is not free: Random Forest gained weighted F1 but lost 0.020 ROC-AUC (fully grown trees), the MLP lost 0.006
  weighted F1 while gaining ROC-AUC, and the Decision Tree's default depth was already the best.

## A3. Final model: Gradient Boosting

**Selected model:** `GradientBoostingClassifier(learning_rate=0.05, max_depth=5, n_estimators=300)` with balanced
sample weights — test accuracy 0.837, weighted precision 0.867, weighted recall 0.837, weighted F1 0.845, macro F1
0.804, balanced accuracy 0.844, **ROC-AUC 0.926**.

**Justification.**
1. **The problem is imbalanced, and the minority class is the one of interest.** Accuracy and weighted F1 are
   dominated by the 75% `<=50K` class, so they reward predicting the majority. Balanced accuracy, macro F1 and
   ROC-AUC weigh both classes, and Gradient Boosting is best of all ten models on all three.
2. **Best ranking quality, and it is stable.** Highest ROC-AUC on the test set (0.926) and in 5-fold CV
   (0.927 ± 0.004); the lead over the next model (Bagging, 0.914 ± 0.003) is far larger than the fold-to-fold noise.
   A good ranking means the decision threshold can later be moved to trade recall against precision as needed.
3. **Bagging's advantage is small.** It leads by 0.005 weighted F1 and 0.009 accuracy, while Gradient Boosting finds
   about 215 more of the 2,240 high earners.
4. **Tuning applied and improvement documented.** Stratified 5-fold `GridSearchCV` over `n_estimators`,
   `learning_rate` and `max_depth` improved every test metric over the default model: accuracy +0.017, weighted F1
   +0.016, macro F1 +0.018, balanced accuracy +0.014, ROC-AUC +0.009.

**Trade-off and alternative.** Gradient Boosting makes about 300 more false `>50K` predictions than Bagging
(precision 0.625 vs 0.665). If the goal were the highest overall accuracy with fewer false alarms, Bagging
(accuracy 0.846, weighted F1 0.849) would be the choice.

Saved: `results/classification/all_classifiers_comparison.csv`, `all_classifiers_tuning_summary.csv`,
`all_classifiers_comparison.png`, `final_model_candidates.png`.